# sanpier kit, fold-safe retrain (Issue #54)
Kit config verbatim; prepare.py gets the normalized_smiles alias and drops rows whose inchikey14 is in fold-safe split.parquet (hold+np) or the v3 holdout answers, plus their tautomer kit keys.


In [ ]:
SANPIER_FOLD0_SRC = '"""sanpier キット（casmi-spectrum-model-v1）を、検証の正解構造を外して学習するための部品（Issue #54）。\n\nキットへの変更は 2 点だけ:\n  1. SMILES 列の別名に normalized_smiles を足す（キットの説明文にある互換の直し、v80 と同じ 1 行）\n  2. 行選択: 競技の inchikey14 が DROP_IK に入る行と、キットの構造キーが DROP_KIT に入る行を落とす\n\nカーネル（kernels/sanpier_fold0）はこのファイルをそのまま埋め込んで使う。\n"""\nimport json\n\nALIAS_OLD = "\'smiles\':[\'smiles\',\'canonical_smiles\']"\nALIAS_NEW = "\'smiles\':[\'smiles\',\'canonical_smiles\',\'normalized_smiles\']"\n\n# 行ごとの最初の処理。ここで競技の inchikey14 で落とす。\nROW_OLD = "total+=1;s=str(row[mapping[\'smiles\']])"\nROW_NEW = ("total+=1;s=str(row[mapping[\'smiles\']]);_ik=str(row.get(\'inchikey14\',\'\'))\\n"\n           "                    if source_index==0 and _ik in DROP_IK:reject[\'dropped_heldout_inchikey14\']+=1;continue")\n# キットの構造キーができた直後。ここで互変異性体のキーで落とす。\nKEY_OLD = "key,canonical,formula,mass,fp=st"\nKEY_NEW = ("key,canonical,formula,mass,fp=st\\n"\n           "                        if key in DROP_KIT:reject[\'dropped_heldout_kitkey\']+=1;continue")\nACC_OLD = "accepted+=1"\nACC_NEW = "accepted+=1;TRAIN_IK.add(_ik)"\nHEADER = "\\nDROP_IK=set();DROP_KIT=set();TRAIN_IK=set()\\n"\n\n\ndef patch_prepare_source(src: str) -> str:\n    """キットの prepare.py の文字列に 2 点の変更を当てる。各アンカーはちょうど 1 回現れることを確かめる。"""\n    for old in (ALIAS_OLD, ROW_OLD, KEY_OLD, ACC_OLD):\n        n = src.count(old)\n        if n != 1:\n            raise ValueError(f"anchor must appear exactly once: {old!r} ({n})")\n    out = src.replace(ALIAS_OLD, ALIAS_NEW).replace(ROW_OLD, ROW_NEW).replace(KEY_OLD, KEY_NEW).replace(ACC_OLD, ACC_NEW)\n    # モジュールの大域変数として置く（import 行の後、ALIASES の前）。\n    i = out.index("ALIASES=")\n    return out[:i] + HEADER.lstrip("\\n") + out[i:]\n\n\ndef drop_inchikey14(split_keys, holdout_keys: dict) -> set:\n    """split.parquet の全行の inchikey14 ∪ 各検証の正解 inchikey14。"""\n    out = set(map(str, split_keys))\n    for keys in holdout_keys.values():\n        out |= set(map(str, keys))\n    return out\n\n\ndef leak_audit(train_ik, split_frame_rows, holdout_keys: dict) -> dict:\n    """学習に使った inchikey14 と、外すべき各集合の重なり件数。split_frame_rows は (inchikey14, fold) の並び。"""\n    train_ik = set(map(str, train_ik))\n    folds = {}\n    for ik, fold in split_frame_rows:\n        folds.setdefault(str(fold), set()).add(str(ik))\n    out = {f"split_{f}": len(train_ik & ks) for f, ks in sorted(folds.items())}\n    for name, keys in sorted(holdout_keys.items()):\n        out[name] = len(train_ik & set(map(str, keys)))\n    out["total"] = sum(out.values())\n    return out\n\n\nif __name__ == "__main__":  # pragma: no cover\n    print(json.dumps({"alias": ALIAS_NEW}))\n'
HOLDOUTS = ['v3_c1', 'v3_c2', 'enveda_v1', 'v3_c2n']
EXPECTED_DROP = 55369
KEYS_FILE = 'sanpier_fold0_holdout_keys.json'

# ---- Issue #54: sanpier kit trained without fold-safe hold/np structures and v3 holdout answers ----
import json, sys, time, hashlib, shutil, subprocess, threading
from pathlib import Path
T0 = time.time()
log = lambda *a: print(*a, flush=True)
def _heartbeat():
    while True:
        time.sleep(300); log(f'HEARTBEAT {round(time.time() - T0)}s')
threading.Thread(target=_heartbeat, daemon=True).start()
import torch
log('ENV python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
try:
    from rdkit import rdBase
except ImportError:
    wh = sorted(Path('/kaggle/input').rglob('rdkit-*.whl')); assert wh, 'rdkit wheel dataset not attached'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(wh[0].parent), 'rdkit==2026.03.3'], check=True)
    from rdkit import rdBase
log('RDKIT', rdBase.rdkitVersion)
assert rdBase.rdkitVersion == '2026.03.3', rdBase.rdkitVersion  # same as v80 and the kit's environment.json

In [ ]:
hits = sorted(Path('/kaggle/input').rglob('casmi_spectrum_model_v1/train.py')); assert len(hits) == 1, hits
KIT = hits[0].parent
STAGE = Path('/kaggle/working/kit_staged'); shutil.copytree(KIT, STAGE, dirs_exist_ok=True)
for _p in STAGE.glob('*.pt'):
    _p.unlink()  # published checkpoints are not used here
(STAGE / 'sanpier_fold0.py').write_text(SANPIER_FOLD0_SRC)
sys.path.insert(0, str(STAGE))
import sanpier_fold0 as sf
_src = (STAGE / 'prepare.py').read_text()
(STAGE / 'prepare.py').write_text(sf.patch_prepare_source(_src))
log('KIT_DELTA prepare.py sha256 before', hashlib.sha256(_src.encode()).hexdigest()[:16],
    'after', hashlib.sha256((STAGE / 'prepare.py').read_bytes()).hexdigest()[:16])
import prepare, train, features

In [ ]:
import pandas as pd, pyarrow.parquet as pq
_tp = sorted(Path('/kaggle/input').rglob('train.parquet')); assert len(_tp) == 1, _tp
TRAIN = _tp[0]
_sp = [p for p in Path('/kaggle/input').rglob('split.parquet') if 'fold-safe' in str(p)]; assert len(_sp) == 1, _sp
SPLIT = pd.read_parquet(_sp[0], columns=['inchikey14', 'fold'])
_kf = sorted(Path('/kaggle/input').rglob(KEYS_FILE)); assert len(_kf) == 1, _kf
HOLDOUT_KEYS = json.loads(_kf[0].read_text())
assert sorted(HOLDOUT_KEYS) == sorted(HOLDOUTS), sorted(HOLDOUT_KEYS)
DROP_IK = sf.drop_inchikey14(SPLIT.inchikey14, HOLDOUT_KEYS)
log('DROP_IK', len(DROP_IK), 'split rows', len(SPLIT), SPLIT.fold.value_counts().to_dict())
assert len(DROP_IK) == EXPECTED_DROP, len(DROP_IK)
# kit structure keys (stereo-stripped, tautomer-canonical) of every dropped row, so a tautomer twin is dropped too
_smi = set()
for _b in pq.ParquetFile(TRAIN).iter_batches(batch_size=65536, columns=['inchikey14', 'normalized_smiles']):
    _d = _b.to_pandas()
    _smi.update(_d.normalized_smiles[_d.inchikey14.isin(DROP_IK)].astype(str))
DROP_KIT, _bad = set(), 0
for _s in sorted(_smi):
    try:
        DROP_KIT.add(features.structure(_s, 2048)[0])
    except (ValueError, RuntimeError):
        _bad += 1
log('DROP_KIT', len(DROP_KIT), 'from smiles', len(_smi), 'invalid', _bad, round(time.time() - T0), 's')
prepare.DROP_IK.update(DROP_IK); prepare.DROP_KIT.update(DROP_KIT)

In [ ]:
cfg = json.loads((KIT / 'config.json').read_text())
cfg.update(train_path=str(TRAIN), work='/kaggle/working/cache_fold0', output='/kaggle/working/casmi_spectrum_model_fold0')
log('CFG', json.dumps(cfg))
audit = prepare.prepare(cfg)
WORK, OUT = Path(cfg['work']), Path(cfg['output']); OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(WORK / 'data_audit.json', OUT / 'data_audit.json'); log('PREP_DONE', round(time.time() - T0), 's')
_kitkeys = set(pd.read_parquet(WORK / 'molecules.parquet', columns=['key']).key)
result = train.run(cfg)
log('TRAIN_RESULT', json.dumps(result))

In [ ]:
TRAIN_IK = set(prepare.TRAIN_IK)
(OUT / 'train_keys.txt').write_text('\n'.join(sorted(TRAIN_IK)) + '\n')
(OUT / 'drop_keys_union.txt').write_text('\n'.join(sorted(DROP_IK)) + '\n')
LEAK = sf.leak_audit(TRAIN_IK, zip(SPLIT.inchikey14, SPLIT.fold), HOLDOUT_KEYS)
DROP_AUDIT = dict(issue=54, drop_inchikey14=len(DROP_IK), drop_kitkey=len(DROP_KIT), train_inchikey14=len(TRAIN_IK),
                  train_kitkey=len(_kitkeys), kitkey_overlap=len(_kitkeys & DROP_KIT), rejected=audit['rejected'],
                  leak=LEAK, seconds=round(time.time() - T0))
(OUT / 'drop_audit.json').write_text(json.dumps(DROP_AUDIT, indent=2))
(OUT / 'config_used.json').write_text(json.dumps(cfg, indent=2))
for _n in ('model.py', 'features.py', 'inference.py', 'prepare.py', 'sanpier_fold0.py'):
    shutil.copy2(STAGE / _n, OUT / _n)
log('DROP_AUDIT', json.dumps(DROP_AUDIT))
assert LEAK['total'] == 0 and DROP_AUDIT['kitkey_overlap'] == 0, DROP_AUDIT
for _p in sorted(OUT.iterdir()):
    log('ARTIFACT', _p.name, _p.stat().st_size, hashlib.sha256(_p.read_bytes()).hexdigest()[:16])
shutil.rmtree(WORK, ignore_errors=True); shutil.rmtree(STAGE, ignore_errors=True)
log(f'SANPIER_FOLD0_DONE {round(time.time() - T0)}s')